In [0]:
import pandas as pd
from pyspark.sql import SparkSession
import numpy as np

spark = SparkSession.builder.getOrCreate()

# mortality table에서 INSEE 2019만 사용
df_mort = (
    spark.table("workspace.life_insurance_raw.mortality_assumption")
    .filter("source = 'INSEE' AND year = 2019")
    .toPandas()
)

# discount curve
df_disc = spark.table("workspace.life_insurance_raw.discount_curve").toPandas()

# cohort input
df_cohort = spark.table("workspace.life_insurance_raw.policy_cohort_input").toPandas()


In [0]:
display(df_mort)

In [0]:
# annual DF: v^t = 1 / (1 + r_t)^t
df_disc["tenor_year"] = df_disc["tenor_month"] / 12
df_disc = df_disc[df_disc["tenor_year"] <= 40]  # safety
df_disc["df"] = 1 / (1 + df_disc["zero_rate_annual"]) ** df_disc["tenor_year"]


In [0]:
display(df_mort)

In [0]:
def compute_net_premium(issue_age, sex, term_years, sum_assured, df_mort, df_disc):
    # 필요한 mortality subset
    ages = list(range(issue_age, issue_age + term_years))
    mort = df_mort[(df_mort["age"].isin(ages)) & (df_mort["sex"] == sex)].sort_values("age")
    
    # 필요한 discount factor subset
    disc = df_disc[df_disc["tenor_year"].isin(range(1, term_years + 1))].sort_values("tenor_year")

    # survival probability: p_0 = 1, p_1 = (1-q_35), p_2 = (1-q_35)(1-q_36), ...
    q_vals = mort["qx_annual"].values  # q_35, q_36, ..., q_64 (term개)
    
    # survival probability {}_t p_x
    p = 1.0
    survival = []
    for q in q_vals:
        survival.append(p)
        p *= (1 - q)
    
    survival = np.array(survival)  # len = term
    v = disc["df"].values          # len = term
    
    # numerator: SA * Σ v^t * p_t * q_{x+t}
    numerator = sum_assured * (v * survival * q_vals).sum()
    
    # denominator: Σ v^t * p_t
    denominator = (v * survival).sum()
    
    return numerator / denominator


In [0]:
display(df_cohort)

In [0]:
rows = []
VERSION_ID = "PRICING_2026_04"

for _, row in df_cohort.iterrows():
    cohort_id = row["cohort_id"]
    issue_age = row["issue_age"]
    sex = row["sex"]
    term = row["term_years"]
    SA = row["sum_assured"]
    
    net_premium = compute_net_premium(issue_age, sex, term, SA, df_mort, df_disc)
    expense_loading = 0.03      # 3% maintenance expense (설계서와 일치)
    profit_margin = 0.02        # 2% profit margin
    contingency_margin = 0.02   # 2% contingency/adverse deviation

    total_loading = expense_loading + profit_margin + contingency_margin  # 7%

    gross_premium = net_premium / (1 - total_loading)
    
    rows.append({
        "cohort_id": cohort_id,
        "sex": sex,
        "annual_net_premium": net_premium,
        "annual_gross_premium": gross_premium,
        "pricing_assumption_version": VERSION_ID,
        "premium_frequency": "ANNUAL",
        "premium_currency": "EUR"
    })

df_premium = pd.DataFrame(rows)


In [0]:
display(df_premium)

In [0]:
df_spark = spark.createDataFrame(df_premium)

(
    df_spark
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.life_insurance_raw.premium_input")
)
